##CODE

In [ ]:
#Install dependencies
!pip install -q timm pillow

#Import libraries
import torch
import timm
import numpy as np
from PIL import Image
import torchvision.transforms as T
from pathlib import Path
import glob
from tqdm import tqdm
import pandas as pd


In [ ]:
#mount google drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
#config
MODEL_PATH = '/content/drive/MyDrive/glenda_splits/edgenext_small/fold1_best.pt'
IMAGE_FOLDER = '/content/drive/MyDrive/testimages/'                     #Folder with  new test images

TARGET_SIZE = (320, 320)
NORM_MEAN = [0.5, 0.5, 0.5]
NORM_STD = [0.5, 0.5, 0.5]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

#transforms
eval_transforms = T.Compose([
    T.Resize(TARGET_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

Using device: cuda


In [ ]:
#build model
def build_model():
    model = timm.create_model('edgenext_small', pretrained=False, num_classes=2)
    return model

model = build_model().to(DEVICE)

# Load trained weights
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint)
model.eval()

print(f"Model loaded successfully from: {MODEL_PATH}")

Model loaded successfully from: /content/drive/MyDrive/glenda_splits/edgenext_small/fold1_best.pt


In [ ]:
#inference function
@torch.no_grad()
def predict_image(image_path):
    try:
        img = Image.open(image_path).convert('RGB')
        tensor = eval_transforms(img).unsqueeze(0).to(DEVICE)

        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred_class = probs.argmax().item()
        confidence = probs[pred_class].item()

        return {
            'image': Path(image_path).name,
            'prediction': 'Pathological (Endometriosis)' if pred_class == 1 else 'Normal',
            'confidence': f"{confidence:.4f}",
            'prob_normal': f"{probs[0].item():.4f}",
            'prob_pathological': f"{probs[1].item():.4f}"
        }
    except Exception as e:
        return {'image': Path(image_path).name, 'error': str(e)}

In [ ]:
#running inference
#Get all images (png, jpg, jpeg, webp)
image_paths = []
for ext in ['*.png', '*.jpg', '*.jpeg', '*.webp', '*.PNG', '*.JPG', '*.JPEG']:
    image_paths.extend(glob.glob(str(Path(IMAGE_FOLDER) / ext)))

print(f"Found {len(image_paths)} images")

results = []

for img_path in tqdm(image_paths):
    result = predict_image(img_path)
    results.append(result)

# ========================== DISPLAY RESULTS ==========================
df = pd.DataFrame(results)

#display
if 'error' in df.columns:
    print("\nFailed images:")
    print(df[df['error'].notna()][['image', 'error']])

print("\nINFERENCE RESULTS:")
print(df[['image', 'prediction', 'confidence']].to_string(index=False))

#Save results
output_csv = '/content/drive/MyDrive/externalval.csv'
df.to_csv(output_csv, index=False)
print(f"\n Results saved to: {output_csv}")

Found 14 images


100%|██████████| 14/14 [00:09<00:00,  1.46it/s]


INFERENCE RESULTS:
                    image                   prediction confidence
                2akXd.jpg Pathological (Endometriosis)     0.9927
                A7CKv.jpg Pathological (Endometriosis)     0.9770
                qVsRQ.jpg Pathological (Endometriosis)     0.7355
endo_subtle_endosalp.webp                       Normal     0.8655
      endo_subtle_it.webp Pathological (Endometriosis)     0.9919
     endo_subtle5_it.webp                       Normal     0.7612
       endo_typical2.webp Pathological (Endometriosis)     0.9944
       endo_typical5.webp Pathological (Endometriosis)     0.9904
  endo_typical-large.webp Pathological (Endometriosis)     0.9884
   endo_typ-white_it.webp Pathological (Endometriosis)     0.9901
  endo_typical4_deep.webp Pathological (Endometriosis)     0.9817
          endo_deep1.webp Pathological (Endometriosis)     0.9665
  endo_deep_sigmoid1.webp Pathological (Endometriosis)     0.9831
                  35.webp Pathological (Endometriosis)  

All 14 of these were endometriosis, which was correctly predicted by around 86%  despite the model being trained on an imbalance of healthy images

Explainability Heatmap (GradCAM++)

Explainability

In [ ]:
#imports
!pip install -q grad-cam

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import cv2

In [ ]:
#setting target layer
target_layers = [model.stages[3].blocks[2]]

In [ ]:
#CAM generation function
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

cam = GradCAM(model=model, target_layers=target_layers)

def generate_cam(image_path, output_folder):
    #Load and preprocess (same as predict_image)
    img = Image.open(image_path).convert('RGB')
    orig = np.array(img) / 255.0  # for overlay, keep as float in [0,1]
    tensor = eval_transforms(img).unsqueeze(0).to(DEVICE)

    #Forward pass to get predicted class
    logits = model(tensor)
    probs = torch.softmax(logits, dim=1)[0]
    pred_class = probs.argmax().item()

    targets = [ClassifierOutputTarget(pred_class)]
    grayscale_cam = cam(input_tensor=tensor, targets=targets)[0]  # H x W

    #Resize CAM to original image size
    grayscale_cam_resized = cv2.resize(grayscale_cam, (orig.shape[1], orig.shape[0]))
    visualization = show_cam_on_image(orig.astype(np.float32), grayscale_cam_resized, use_rgb=True)

    img_name = Path(image_path).stem
    output_path = Path(output_folder) / f"{img_name}_cam.png"
    cv2.imwrite(str(output_path), cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR))

    return str(output_path), pred_class, float(probs[pred_class].item())

In [ ]:
#running on all test images
OUTPUT_CAM_FOLDER = "/content/drive/MyDrive/testimages_cams"
Path(OUTPUT_CAM_FOLDER).mkdir(parents=True, exist_ok=True)

cam_results = []

for img_path in tqdm(image_paths):
    cam_path, pred_class, pred_conf = generate_cam(img_path, OUTPUT_CAM_FOLDER)
    cam_results.append({
        "image": Path(img_path).name,
        "cam_image": Path(cam_path).name,
        "pred_class": int(pred_class),
        "pred_conf": pred_conf,
    })

cam_df = pd.DataFrame(cam_results)
cam_csv = "/content/drive/MyDrive/externalvalidation_cams.csv"
cam_df.to_csv(cam_csv, index=False)
print(f"CAMs saved in folder: {OUTPUT_CAM_FOLDER}")
print(f"CAM metadata saved to: {cam_csv}")

100%|██████████| 14/14 [00:02<00:00,  6.92it/s]

CAMs saved in folder: /content/drive/MyDrive/testimages_cams
CAM metadata saved to: /content/drive/MyDrive/externalvalidation_cams.csv
